In [2]:
import pandas as pd
import os
import glob
import zipfile

path = "/Users/saraseid/Desktop/2022-citibike-tripdata"
zip_files = glob.glob(os.path.join(path, "*.zip"))
print("ZIP files found:", len(zip_files))

dfs = []
for z in zip_files:
    with zipfile.ZipFile(z, "r") as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        print(os.path.basename(z), "->", len(csv_names), "csv")
        for name in csv_names:
            with zf.open(name) as f:
                dfs.append(pd.read_csv(f, low_memory=False))

print("Dataframes loaded:", len(dfs))

df = pd.concat(dfs, ignore_index=True)
print("Final shape:", df.shape)

df.head()


ZIP files found: 12
202209-citibike-tripdata.zip -> 4 csv
202201-citibike-tripdata.zip -> 2 csv
202203-citibike-tripdata.zip -> 2 csv
202205-citibike-tripdata.zip -> 3 csv
202207-citibike-tripdata.zip -> 4 csv
202211-citibike-tripdata.zip -> 3 csv
202208-citibike-tripdata.zip -> 4 csv
202202-citibike-tripdata.zip -> 2 csv
202212-citibike-tripdata.zip -> 2 csv
202204-citibike-tripdata.zip -> 3 csv
202210-citibike-tripdata.zip -> 3 csv
202206-citibike-tripdata.zip -> 4 csv
Dataframes loaded: 36
Final shape: (29838806, 13)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,3A2034DA67C8874B,electric_bike,2022-09-14 20:37:39.155,2022-09-14 21:05:39.760,N 12 St & Bedford Ave,5450.04,Albany St & Greenwich St,5145.02,40.720796,-73.954882,40.709267,-74.013247,casual
1,F1C63DACDCC1C33D,electric_bike,2022-09-17 19:21:39.407,2022-09-17 20:08:32.670,Union Ave & Jackson St,5300.06,Metropolitan Ave & Meeker Ave,5300.05,40.716075,-73.952029,40.714133,-73.952344,casual
2,94372C52777F4AC1,electric_bike,2022-09-17 16:49:21.830,2022-09-17 17:01:51.537,S 4 St & Wythe Ave,5204.05,St Marks Pl & 1 Ave,5626.13,40.712874,-73.965935,40.727791,-73.985649,casual
3,44818FEC94B62B66,electric_bike,2022-09-08 12:27:40.019,2022-09-08 12:47:15.649,Amsterdam Ave & W 73 St,7260.09,Washington St & Gansevoort St,6039.06,40.779668,-73.980930,40.739323,-74.008119,casual
4,F8A63709F214EBAA,classic_bike,2022-09-16 19:00:19.266,2022-09-16 19:07:28.905,University Pl & E 14 St,5905.14,Washington St & Gansevoort St,6039.06,40.734814,-73.992085,40.739323,-74.008119,member


In [ ]:
df.to_csv("citibike_2022_full.csv", index=False)

# This code reads multiple CSV files stored inside ZIP archives and combines
# them into one dataframe using pd.concat(), which is the most efficient method
# for joining datasets with the same structure.

In [1]:
token= "jPmaNYsEkQqEpwyHmfydLfdhcIbDfAdC"

In [7]:
import requests
import pandas as pd

BASE_URL = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
STATION_ID = "GHCND:USW00014732"
START_DATE = "2022-01-01"
END_DATE = "2022-12-31"

all_results = []
offset = 1
limit = 1000

while True:
    params = [
        ("datasetid", "GHCND"),
        ("stationid", STATION_ID),
        ("startdate", START_DATE),
        ("enddate", END_DATE),
        ("datatypeid", "TMIN"),
        ("datatypeid", "TMAX"),
        ("datatypeid", "PRCP"),
        ("datatypeid", "AWND"),
        ("units", "metric"),
        ("limit", limit),
        ("offset", offset),
    ]

    r = requests.get(BASE_URL, headers={"token": TOKEN}, params=params)
    r.raise_for_status()
    data = r.json()

    results = data.get("results", [])
    all_results.extend(results)

    meta = data.get("metadata", {}).get("resultset", {})
    count = meta.get("count", 0)

    if offset + limit > count:
        break

    offset += limit

df_long = pd.DataFrame(all_results)
df_long["date"] = pd.to_datetime(df_long["date"]).dt.date

df_weather = (
    df_long.pivot_table(index="date", columns="datatype", values="value", aggfunc="first")
    .reset_index()
)

df_weather.to_csv("lga_weather_2022.csv", index=False)
df_weather.head()

datatype,date,AWND,PRCP,TMAX,TMIN
0,2022-01-01,2.8,19.3,13.9,10.0
1,2022-01-02,4.3,1.0,15.6,3.9
2,2022-01-03,6.4,0.0,3.9,-4.3
3,2022-01-04,3.9,0.0,2.2,-6.0
4,2022-01-05,3.4,6.1,8.9,0.0


In [11]:
import pandas as pd


df_bike = pd.read_csv("citibike_2022_full.csv", low_memory=False)
df_weather = pd.read_csv("lga_weather_2022.csv")

# create date column from bike start time
df_bike["started_at"] = pd.to_datetime(df_bike["started_at"])
df_bike["date"] = df_bike["started_at"].dt.date


df_weather["date"] = pd.to_datetime(df_weather["date"]).dt.date

# merge datasets
df_merged = df_bike.merge(df_weather, how="left", on="date")

# export merged dataset
df_merged.to_csv("citibike_2022_merged_weather.csv", index=False)

df_merged.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,date,AWND,PRCP,TMAX,TMIN
0,3A2034DA67C8874B,electric_bike,2022-09-14 20:37:39.155,2022-09-14 21:05:39.760,N 12 St & Bedford Ave,5450.04,Albany St & Greenwich St,5145.02,40.720796,-73.954882,40.709267,-74.013247,casual,2022-09-14,5.2,0.0,26.7,18.3
1,F1C63DACDCC1C33D,electric_bike,2022-09-17 19:21:39.407,2022-09-17 20:08:32.670,Union Ave & Jackson St,5300.06,Metropolitan Ave & Meeker Ave,5300.05,40.716075,-73.952029,40.714133,-73.952344,casual,2022-09-17,4.2,0.0,23.3,18.3
2,94372C52777F4AC1,electric_bike,2022-09-17 16:49:21.830,2022-09-17 17:01:51.537,S 4 St & Wythe Ave,5204.05,St Marks Pl & 1 Ave,5626.13,40.712874,-73.965935,40.727791,-73.985649,casual,2022-09-17,4.2,0.0,23.3,18.3
3,44818FEC94B62B66,electric_bike,2022-09-08 12:27:40.019,2022-09-08 12:47:15.649,Amsterdam Ave & W 73 St,7260.09,Washington St & Gansevoort St,6039.06,40.779668,-73.980930,40.739323,-74.008119,casual,2022-09-08,3.5,0.0,25.6,18.3
4,F8A63709F214EBAA,classic_bike,2022-09-16 19:00:19.266,2022-09-16 19:07:28.905,University Pl & E 14 St,5905.14,Washington St & Gansevoort St,6039.06,40.734814,-73.992085,40.739323,-74.008119,member,2022-09-16,2.6,0.0,25.0,15.0


In [12]:
df_merged.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'date', 'AWND', 'PRCP', 'TMAX', 'TMIN'],
      dtype='object')

In [13]:
df_merged.to_csv("citibike_2022_merged_weather.csv", index=False)